# Lab 0a · PyTorch from zero

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omar-florez/training-efficient-llms/blob/main/labs/lab_pytorch.ipynb)

Everything in this book — every 70B model, every training trick — is built out of the ~30 PyTorch operations in this notebook. It assumes you can write Python and nothing else. Run it top to bottom **before Chapter 1's lab**; by the end you will have trained a real (tiny) neural network and checked backprop against your own hand computation.

📖 Companion program: [Training Efficient LLMs](https://omar-florez.github.io/training-efficient-llms/) · interactive [forward & backprop stepper](https://omar-florez.github.io/training-efficient-llms/interactive/backprop.html)

In [ ]:
import numpy as np, matplotlib.pyplot as plt

NAVY, BLUE, LIGHT, AMBER, GRAY = "#17406b", "#3f74b8", "#9dbfe4", "#b45309", "#5c5c5c"
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white","axes.edgecolor":GRAY,
    "axes.labelcolor":"#1a1a1a","axes.grid":True,"grid.color":"#e3eaf3","grid.linewidth":0.8,
    "axes.spines.top":False,"axes.spines.right":False,"font.size":11,"figure.dpi":110})
print("ready")

In [ ]:
import torch
print(torch.__version__)
print("GPU available:", torch.cuda.is_available())   # in Colab: Runtime -> Change runtime type -> T4 GPU

## 1 · Tensors: the only data structure

A tensor is an n-dimensional array plus three pieces of metadata that will follow you through the whole book: **shape** (how many numbers, arranged how), **dtype** (how many bytes each number takes — this is where fp32/bf16 memory math comes from), and **device** (which chip the bytes live on).

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])          # from a Python list
B = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])            # 2x3 matrix
z = torch.zeros(2, 3)                       # like np.zeros
r = torch.randn(2, 3)                       # gaussian noise -- how every weight is born
seq = torch.arange(10)                      # 0..9

for t, name in [(a,"a"), (B,"B"), (r,"r"), (seq,"seq")]:
    print(f"{name}: shape={tuple(t.shape)}  dtype={t.dtype}  device={t.device}")

In [ ]:
# dtype is memory: one number = 4 bytes in float32, 2 in bfloat16
x32 = torch.randn(1000, 1000)               # default float32
x16 = x32.to(torch.bfloat16)
print(x32.element_size(), "bytes/number ->", x32.numel()*x32.element_size()/1e6, "MB")
print(x16.element_size(), "bytes/number ->", x16.numel()*x16.element_size()/1e6, "MB")
# a 7B-parameter model in bf16: 7e9 * 2 bytes = 14 GB just for weights

## 2 · Shapes: reshape, slice, combine

Debugging a model *is* debugging shapes. The operations below cover 95% of what you will do.

In [ ]:
t = torch.arange(24.)
M = t.reshape(4, 6)          # same 24 numbers, new view
print(M)
print("row 0:", M[0])
print("col 0:", M[:, 0])
print("block:", M[1:3, 2:4].shape)
print("transpose:", M.T.shape)                    # (6, 4)
print("as 2x3x4:", M.reshape(2, 3, 4).shape)      # tensors are often 3-D+: (batch, seq, dim)

In [ ]:
u = torch.randn(3)
print("unsqueeze:", u.unsqueeze(0).shape, u.unsqueeze(1).shape)   # add a dimension: (1,3) / (3,1)
two = torch.stack([u, u])                                          # new dim: (2, 3)
cat = torch.cat([M, M], dim=1)                                     # glue along existing dim: (4, 12)
print("stack:", two.shape, " cat:", cat.shape)

## 3 · Elementwise math and broadcasting

Operations apply to every element at once (never write a Python loop over tensor elements — that is the whole point). **Broadcasting** stretches a smaller tensor across a bigger one when shapes are compatible: line the shapes up from the right; each pair of dims must be equal or one of them 1.

In [ ]:
X = torch.randn(4, 3)
print((X * 2 + 1).shape)            # scalar broadcasts everywhere
row = torch.tensor([10., 20., 30.]) # shape (3,)
print((X + row).shape)              # (4,3)+(3,) -> row added to every row
col = torch.randn(4, 1)
print((X * col).shape)              # (4,3)*(4,1) -> col scales every column entry

print("sum all:", X.sum().item())
print("mean of each column:", X.mean(dim=0))     # reduce over rows -> shape (3,)
print("max of each row:", X.max(dim=1).values)   # reduce over cols -> shape (4,)

In [ ]:
# softmax by hand -- the same three lines inside every attention layer (Chapter 3)
scores = torch.tensor([2.0, 1.0, 0.1])
p = torch.exp(scores) / torch.exp(scores).sum()
print("by hand:", p)
print("built-in:", torch.softmax(scores, dim=0))

## 4 · Matrix multiplication — the operation the GPU exists for

`@` multiplies matrices. Roughly 99% of the FLOPs in training an LLM are this one op, so it deserves three cells: chaining several matrices, batching many multiplications at once, and counting the cost.

In [ ]:
A = torch.randn(4, 5)
Bm = torch.randn(5, 6)
C = torch.randn(6, 2)

Y = A @ Bm                     # (4,5)@(5,6) -> (4,6): inner dims must match
chain = A @ Bm @ C             # (4,5)@(5,6)@(6,2) -> (4,2): a 3-"layer" stack of transforms
print(Y.shape, chain.shape)

# cost of (m,n)@(n,p): 2*m*n*p FLOPs  (multiply + add per cell)
m, n, p = A.shape[0], A.shape[1], Bm.shape[1]
print(f"A@Bm costs 2*{m}*{n}*{p} = {2*m*n*p} FLOPs")

In [ ]:
# batched matmul: multiply MANY matrix pairs in one call.
# This is how a batch of 32 sentences goes through a layer together.
Qb = torch.randn(32, 128, 64)      # 32 independent (128,64) matrices
Kb = torch.randn(32, 64, 128)
S  = Qb @ Kb                       # -> (32, 128, 128): 32 matmuls at once
print(S.shape)

# einsum: name the axes, let PyTorch wire the multiply. Same result:
S2 = torch.einsum("bik,bkj->bij", Qb, Kb)
print("identical:", torch.allclose(S, S2))

In [ ]:
# a language-model layer is just: look up vectors, multiply by a weight matrix
vocab, d = 50000, 512
E   = torch.randn(vocab, d)                # embedding table (Chapter 3)
ids = torch.tensor([[11, 4022, 9]])        # a 3-token sentence, as ids
h   = E[ids]                               # lookup -> (1, 3, 512)
W   = torch.randn(d, d)
out = h @ W                                # one linear layer -> (1, 3, 512)
print(h.shape, "->", out.shape)

## 5 · Devices: moving to the GPU

The model, the data, and the math must all be on the same device. `.to(device)` is the only API you need — and the pattern below (`cuda` if available, else CPU) is in every script in this book.

In [ ]:
import time
device = "cuda" if torch.cuda.is_available() else "cpu"
print("using:", device)

big1, big2 = torch.randn(2048, 2048), torch.randn(2048, 2048)
t0 = time.time(); big1 @ big2; cpu_ms = (time.time()-t0)*1000

g1, g2 = big1.to(device), big2.to(device)
g1 @ g2                                    # warm-up
if device == "cuda": torch.cuda.synchronize()
t0 = time.time(); g1 @ g2
if device == "cuda": torch.cuda.synchronize()
dev_ms = (time.time()-t0)*1000
print(f"2048x2048 matmul: cpu {cpu_ms:.1f} ms | {device} {dev_ms:.2f} ms")

## 6 · Autograd: the gradient, for free

Chapter 1 computes this example by hand (and the [backprop stepper](https://omar-florez.github.io/training-efficient-llms/interactive/backprop.html) animates it): model `f = w·x + b`, input x = 2, target y = 7, loss `L = (f − y)²`. The chain rule gives **∂L/∂w = −8** and **∂L/∂b = −4**. Ask PyTorch to check us: mark the parameters with `requires_grad=True`, compute the loss, call `.backward()`.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
x, y = torch.tensor(2.0), torch.tensor(7.0)

f = w * x + b                # forward: 5
L = (f - y) ** 2             # loss: 4
L.backward()                 # backward: fills .grad on every requires_grad tensor

print("L =", L.item())
print("dL/dw =", w.grad.item(), "  dL/db =", b.grad.item())   # -8, -4 -- matches the chapter

In [ ]:
# one gradient-descent step, then zero the grads (they ACCUMULATE if you forget!)
with torch.no_grad():
    w -= 0.1 * w.grad
    b -= 0.1 * b.grad
w.grad.zero_(); b.grad.zero_()
print("after step: w =", w.item(), " b =", b.item(), " new f =", (w*x+b).item(), " (target 7)")

## 7 · `nn.Module`: a real (tiny) network, trained

`nn.Module` packages parameters + forward pass; an optimizer packages the update rule. The five-line loop below — forward, loss, `zero_grad`, `backward`, `step` — is *the same loop* that trains a 70B model. Everything after this notebook is about making it fast.

In [ ]:
import torch.nn as nn

torch.manual_seed(0)
Xtr = torch.linspace(-3, 3, 200).unsqueeze(1)
Ytr = torch.sin(Xtr) + 0.1 * torch.randn_like(Xtr)     # noisy sine wave

model = nn.Sequential(nn.Linear(1, 32), nn.ReLU(),
                      nn.Linear(32, 32), nn.ReLU(),
                      nn.Linear(32, 1))
n_params = sum(p.numel() for p in model.parameters())
print("parameters:", n_params)

opt, loss_fn, hist = torch.optim.Adam(model.parameters(), lr=1e-2), nn.MSELoss(), []
for step in range(600):
    loss = loss_fn(model(Xtr), Ytr)   # forward + loss
    opt.zero_grad()                   # clear old grads
    loss.backward()                   # backward
    opt.step()                        # update
    hist.append(loss.item())
print(f"loss: {hist[0]:.3f} -> {hist[-1]:.3f}")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
a1.semilogy(hist, color=NAVY, lw=1.6); a1.set(xlabel="step", ylabel="MSE (log)", title="The loss curve")
with torch.no_grad(): pred = model(Xtr)
a2.plot(Xtr, Ytr, ".", color=LIGHT, ms=4, label="data")
a2.plot(Xtr, pred, color=AMBER, lw=2.2, label="model")
a2.set(xlabel="x", title=f"What {n_params} parameters learned"); a2.legend()
plt.tight_layout(); plt.show()

## Exercises

1. **Broadcasting**: create a `(5, 1)` and a `(1, 4)` tensor; predict the shape of their sum, then check.
2. **Batched matmul**: make attention-like scores for 8 heads at once — `(8, 16, 64) @ (8, 64, 16)` — then softmax the last dim and verify each row sums to 1.
3. **Autograd**: change the loss to `L = |f − y|` and recompute the gradients. Why is ∂L/∂w now ±x instead of proportional to the error?
4. **Forget `zero_grad()`** in the training loop on purpose. What happens to the loss curve, and why?
5. **Count FLOPs**: for the tiny MLP, compute FLOPs per forward pass by hand (2·m·n·p per layer) and confirm it's ≈ 2 × the parameter count per sample — the rule Chapter 1 uses at 7B scale.